# Part 0: Data Prepare

In [ ]:
pip install wrds pandas numpy

In [ ]:
import wrds
import pandas as pd
import numpy as np

## 1. Connect to WRDS

In [ ]:
db = wrds.Connection()

## 2. Download the Compustat Annual Financial Data

In [ ]:
comp = db.raw_sql("""
SELECT
    f.gvkey,
    f.datadate,
    f.fyear,
    f.fyr,
    f.at,
    f.sale,
    f.ni,
    f.ceq,
    f.cusip,
    f.sich AS sic    
FROM comp.funda f
WHERE
    f.indfmt = 'INDL'
    AND f.datafmt = 'STD'
    AND f.consol = 'C'
    AND f.popsrc = 'D'
    AND f.fyear BETWEEN 2013 AND 2023
""")

In [ ]:
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['cusip8'] = comp['cusip'].astype(str).str[:8]

In [ ]:
# keep total assets > 0
comp = comp[comp['at'] > 0].copy()

In [ ]:
# Construct basic variables
comp['roa']  = comp['ni'] / comp['at']
comp['size'] = np.log(comp['at'])
comp['year'] = comp['datadate'].dt.year

In [ ]:
print(f"Compustat rows: {comp.shape[0]}")

## 3. Download the Monthly CRSP Data

In [ ]:
crsp = db.raw_sql("""
SELECT
    m.permno,
    m.date,
    m.ret,
    m.prc,
    m.shrout,
    e.shrcd,
    e.exchcd
FROM crsp.msf m
INNER JOIN crsp.msenames e
    ON m.permno = e.permno
    AND m.date BETWEEN e.namedt AND COALESCE(e.nameendt, '2099-12-31')
WHERE
    m.date BETWEEN '2013-01-01' AND '2025-12-31'
    AND e.shrcd IN (10, 11)
    AND e.exchcd IN (1, 2, 3)
""")

In [ ]:
crsp['date']   = pd.to_datetime(crsp['date'])
crsp['ret']    = pd.to_numeric(crsp['ret'],    errors='coerce')
crsp['prc']    = pd.to_numeric(crsp['prc'],    errors='coerce')
crsp['shrout'] = pd.to_numeric(crsp['shrout'], errors='coerce')
crsp['me']     = crsp['prc'].abs() * crsp['shrout']
crsp['year']   = crsp['date'].dt.year
crsp['month']  = crsp['date'].dt.month

In [ ]:
print(f"CRSP (after filtering): {crsp.shape[0]}")

In [ ]:
# Retain valid monthly earnings
crsp_ret = crsp[['permno', 'date', 'ret']].dropna(subset=['ret']).copy()

In [ ]:
def compute_post_fy_return(permno_series, datadate_series, crsp_ret_df):
    """
    For each (permno, datadate) record,
    The monthly earnings from the first to the twelfth month after taking the datadate
    Calculate the compound annualized return (1+r1)*(1+r2)*... *(1+r12) -1.
    If the number of months is less than 12, return NaN and also return the actual number of months.
    """
    records = []
    crsp_grouped = crsp_ret_df.groupby('permno')

    for permno, datadate in zip(permno_series, datadate_series):
        if permno not in crsp_grouped.groups:
            records.append({'future_annual_ret': np.nan, 'future_n_months_ret': 0})
            continue

        sub = crsp_grouped.get_group(permno).copy()
        # Take the monthly observations after datadate and sort them by date
        sub = sub[sub['date'] > datadate].sort_values('date')
        # The maximum is 12 months
        sub12 = sub.head(12)
        n = len(sub12)

        if n < 12:
            records.append({'future_annual_ret': np.nan, 'future_n_months_ret': n})
        else:
            compound_ret = (1 + sub12['ret']).prod() - 1
            records.append({'future_annual_ret': compound_ret, 'future_n_months_ret': n})

    return pd.DataFrame(records)

## 4. Download and Clean the CCM Link Table

In [ ]:
ccm = db.raw_sql("""
SELECT
    gvkey,
    lpermno AS permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
FROM crsp.ccmxpf_linktable
WHERE
    linktype IN ('LC', 'LU', 'LS')
    AND linkprim IN ('C', 'P')
""")

In [ ]:
ccm['linkdt']    = pd.to_datetime(ccm['linkdt'])
ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt'])

## 5. Merge Compustat + CCM and Retain the Valid Link Window

In [ ]:
comp_ccm = comp.merge(ccm, on='gvkey', how='left')

In [ ]:
comp_ccm = comp_ccm[
    (comp_ccm['datadate'] >= comp_ccm['linkdt']) &
    (
        (comp_ccm['datadate'] <= comp_ccm['linkenddt']) |
        (comp_ccm['linkenddt'].isna())
    )
].copy()

In [ ]:
print(f"Comp+CCM rows after merge: {comp_ccm.shape[0]}")

## 6. Download ESG Data (item=1 = Overall ESG score)

In [ ]:
dfs = []
for y in range(2013, 2024):
    print(f"download ESG score {y}...")
    tmp = db.raw_sql(f"""
    SELECT
        orgpermid,
        fy,
        value_ AS esg_score
    FROM tr_esg.esgscores
    WHERE item = 1        -- item=1: ESGScore（overall score，not Combined）
      AND fy = {y}
    """)
    dfs.append(tmp)

In [ ]:
esg = pd.concat(dfs, ignore_index=True)
esg = esg.dropna(subset=['orgpermid', 'fy', 'esg_score']).copy()
esg = esg.groupby(['orgpermid', 'fy'], as_index=False)['esg_score'].mean()

In [ ]:
print(f"ESG rows: {esg.shape[0]}")

## 7. Download the ESG Reference Table (obtain CUSIP for integration with Compustat)

In [ ]:
org_list = esg['orgpermid'].dropna().astype('int64').unique().tolist()

In [ ]:
chunks = []
chunk_size = 500
for i in range(0, len(org_list), chunk_size):
    chunk_ids = org_list[i:i+chunk_size]
    id_str = ",".join(map(str, chunk_ids))
    print(f"chunk {i} to {i+len(chunk_ids)}")
    tmp = db.raw_sql(f"""
    SELECT DISTINCT
        orgpermid,
        year,
        cusip,
        isin,
        comname
    FROM tr_esg.wrds_ref_esg
    WHERE year BETWEEN 2013 AND 2023
      AND orgpermid IN ({id_str})
    """)
    chunks.append(tmp)

In [ ]:
esg_ref = pd.concat(chunks, ignore_index=True)
esg_ref['year']   = pd.to_numeric(esg_ref['year'], errors='coerce')
esg_ref['cusip8'] = esg_ref['cusip'].astype(str).str[:8]
esg_ref = esg_ref.dropna(subset=['orgpermid', 'year', 'cusip8']).copy()
esg_ref = esg_ref.drop_duplicates(subset=['orgpermid', 'year', 'cusip8'])

In [ ]:
# merge ESG 
esg_full = esg.merge(
    esg_ref,
    left_on=['orgpermid', 'fy'],
    right_on=['orgpermid', 'year'],
    how='left'
)
esg_full = esg_full.dropna(subset=['cusip8']).copy()

In [ ]:
print(f"ESG full rows: {esg_full.shape[0]}")

## 8. Integrate ESG into the Comp+CCM Panel

In [ ]:
panel = comp_ccm.merge(
    esg_full[['cusip8', 'fy', 'esg_score']].drop_duplicates(),
    left_on=['cusip8', 'fyear'],
    right_on=['cusip8', 'fy'],
    how='left'
)

## 9. Calculate the Future Return (12 months after the end of the fiscal year)

In [ ]:
print("Calculate the future return (aligned to the end date of the fiscal year)...")

In [ ]:
future_ret_df = compute_post_fy_return(
    panel['permno'].values,
    panel['datadate'].values,
    crsp_ret
)

In [ ]:
panel['future_annual_ret']   = future_ret_df['future_annual_ret'].values
panel['future_n_months_ret'] = future_ret_df['future_n_months_ret'].values

## 10. Construct the Future ROA and Lag Control Variables

In [ ]:
panel = panel.sort_values(['gvkey', 'fyear']).copy()

In [ ]:
panel['future_roa'] = panel.groupby('gvkey')['roa'].shift(-1)
panel['lag_roa']    = panel.groupby('gvkey')['roa'].shift(1)
panel['lag_size']   = panel.groupby('gvkey')['size'].shift(1)

## 11. Final Sample Filtering

In [ ]:
panel_final = panel[
    panel['esg_score'].notna() &
    (panel['at'] > 0) &
    (panel['future_n_months_ret'] >= 12)   
].copy()

panel_final['esg_score'] = panel_final['esg_score'].clip(
    lower=panel_final['esg_score'].quantile(0.01),
    upper=panel_final['esg_score'].quantile(0.99)
)

In [ ]:
print(f"final sample rows: {panel_final.shape[0]}")

In [ ]:
dup_check = panel_final.duplicated(subset=['gvkey', 'fyear']).sum()
print(dup_check)

In [ ]:
panel_final.duplicated(subset=['gvkey', 'fyear']).sum()

## 12. Select the Final Column and Save

In [ ]:
final_cols = [
    'gvkey', 'permno', 'datadate', 'fyear', 'cusip8',
    'sic',           
    'at', 'sale', 'ni', 'ceq',
    'roa', 'size',
    'lag_roa', 'lag_size',
    'esg_score',
    'future_roa',
    'future_annual_ret',
    'future_n_months_ret'
]

In [ ]:
final_panel = panel_final[final_cols].copy()
final_panel.to_csv('esg_financial_panel_2013_2023.csv', index=False)

In [ ]:
print("\n=== Descriptive statistics ===")
print(final_panel[['esg_score', 'roa', 'future_roa', 'future_annual_ret', 'at', 'size']].describe())

In [ ]:
print("\n=== Sample size (by year)===")
print(final_panel.groupby('fyear')['gvkey'].count())

In [ ]:
print("\n=== ESG average (Annual)===")
print(final_panel.groupby('fyear')['esg_score'].mean())

In [ ]:
# See how many Cusip8s in esg_full can be found in comp
esg_us = esg_full[esg_full['cusip8'].isin(comp['cusip8'])]
print(f"ESG companies that could be matched with Compustat in 2013: {esg_us[esg_us['fy']==2013].shape[0]}")

esg_ref_2013 = esg_ref[esg_ref['year'] == 2013]
us_firms = esg_ref_2013[esg_ref_2013['isin'].astype(str).str.startswith('US')]
print(f"The number of American companies in 2013: {us_firms.shape[0]}")
print(f"The total number of companies in 2013: {esg_ref_2013.shape[0]}")